In [ ]:
class AFMCantilever:
    def __init__(self, 
                 k: float = 0.5,  # Spring constant [N/m]
                 f0: float = 75e3,  # Resonance frequency [Hz]
                 Q: float = 100,  # Quality factor
                 T: float = 300,
                path: str = 'None'):  # Temperature [K]

        self.k = k
        self.f0 = f0
        self.Q = Q
        self.T = T
        self.kB = 1.380649e-23  # Boltzmann constant [J/K]
        self.path = path
        
        self.m_eff = k / (2 * np.pi * f0)**2
        
        self.gamma = 2 * np.pi * f0 * self.m_eff / Q
        
    def thermal_noise_rms(self) -> float:
        return np.sqrt(self.kB * self.T / self.k)
    
    def thermal_noise_psd_theoretical(self, f: np.ndarray) -> np.ndarray:
        omega = 2 * np.pi * f
        omega0 = 2 * np.pi * self.f0
        
        denominator = (omega0**2 - omega**2)**2 + (omega * omega0 / self.Q)**2
        S_x = (4 * self.kB * self.T * self.gamma) / (self.k**2 * denominator)
        
        return S_x

    def thermal_noise_psd_experimental(self, path: str) -> np.ndarray:
        
        with open(path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            if "sensitivity" in line:
                value = float(line.split(":")[1].split()[0])

        data = np.loadtxt(path, skiprows=24)
        freqs = data[:, 0]
        S_v = data[:, 3]
        S_x = S_v * (value*1e-9)**2
        
        return freqs, S_x

    def generate_thermal_noise(self, n_points: int, fs: float, type: str) -> np.ndarray:

        dt = 1/fs
        df = fs/n_points

        if type == "Theoretical":
            freqs = np.fft.rfftfreq(n_points, dt)
            psd = self.thermal_noise_psd_theoretical(freqs) 
        elif type == "Experimental":
            freqs, psd = self.thermal_noise_psd_experimental(self.path)

        noise = (np.random.normal(size=len(freqs)) +
                 1j * np.random.normal(size=len(freqs)))
        
        amplitudes = np.sqrt(psd * df) * noise
        phase = np.random.uniform(0, 2*np.pi, len(freqs))
        fourier_coeffs = amplitudes * np.exp(1j * phase)
        
        fourier_coeffs[0] = np.abs(fourier_coeffs[0])
        
        noise = np.fft.irfft(amplitudes, n_points)
        
        return noise